In [ ]:
#Basic from scratch decision tree classifier implementation in Python


#Dependencies
import math
import numpy as np
import pandas as pd


In [ ]:
#First we define the node class

class Node():

    def __init__(self,threshhold: float = 0.0, feature: int = 0, Leaf: bool = False, ds_Y: list = [], index:int = 0):

        self.threshold = threshhold
        self.feature = feature
        self.Leaf = Leaf
        self.ds_Y = ds_Y
        self.index = index


        if Leaf == True:
            self.leaf_node()

    def evaluate(self, value):
        value = float(value)

        if self.Leaf is True:
            self.leaf_node()
            return 'leaf'

        if value > float(self.threshold):
            self.next_index_L = (self.index+1)
            return 'left'
        self.next_index_R = self.index+2
        return 'right'

    def leaf_node(self):
        
        ds_y = self.ds_Y
        class_dict: dict = {}
        total_instances: int = int()

        for unique_class in set(ds_y):
            class_dict[unique_class] = 0
        
        for unique_class in class_dict:
            for class_instance in ds_y:
                if class_instance == unique_class:
                    total_instances += 1
                    class_dict[unique_class] +=1
        

        for unique_class in class_dict:
            class_dict[unique_class] = class_dict[unique_class] / total_instances

        self.class_dict = class_dict

        
    
class DecisionTree():

    def __init__(self, max_depth: int = 10):
        self.max_depth: int = max_depth
        self.model: list = []
        pass

    def gini(self, ds_Y: list) -> float:
        '''
        Gini = 1 - Sum((proportion of set)**2)
        '''
        gini:float = 0.0
        gini_dict: dict = {}
        no_elements: float = float(len(ds_Y))
        for value in ds_Y:
            if value not in gini_dict:
                gini_dict[value] = 0
            
            gini_dict[value] += 1.0
        
        for key in gini_dict:
            gini_dict[key] = (gini_dict[key]/no_elements)**2
            gini += gini_dict[key]

        return 1-gini

    def stopping_condition():
        return False

    def split(self, ds_X, ds_y):
        
        best_gini_dict_L: dict = {}
        best_gini_dict_R: dict = {}

        for feature in ds_X:
            
            best_gini_dict_R[ds_X.index(feature)] = [100, 0]
            best_gini_dict_L[ds_X.index(feature)] = [100, 0]
            for threshhold in feature:
                
                threshhold_index = feature.index(threshhold)               

                ds_Y_L = [y for val, y in ds_y[feature] if val < threshhold]
                ds_Y_R = [y for val, y in ds_y[feature] if val >= threshhold]
                gini_L = self.gini(ds_Y_L)
                gini_R = self.gini(ds_Y_R)

                proportion_left = len(ds_Y_L)
                proportion_right = len(ds_Y_R)
                total_proportion = proportion_left+proportion_right
                if total_proportion == 0:
                    break
                    

                weighted_sum = (((gini_L)*(proportion_left/total_proportion))+((gini_R)*((proportion_right/total_proportion))))/2
                current_ws = best_gini_dict_L[ds_X.index(feature)][0]

                if weighted_sum<current_ws:
                    best_gini_dict_L[ds_X.index(feature)] = [weighted_sum, threshhold, threshhold_index, ds_Y_L]
                    best_gini_dict_R[ds_X.index(feature)] = [weighted_sum, threshhold, threshhold_index, ds_Y_R]
            
            highest_list_L: list = [100, 0, 0, []]

            for key in best_gini_dict_L:
                proposed_gini = best_gini_dict_L[key]
                if highest_list_L[0]>proposed_gini[0]:
                    highest_list_L = best_gini_dict_L[key]
            
            highest_list_R: list = [100, 0, 0, []]

            for key in best_gini_dict_R:
                proposed_gini = best_gini_dict_R[key]
                if highest_list_R[0]>proposed_gini[0]:
                    highest_list_R = best_gini_dict_R[key]
        
            return highest_list_L[1], highest_list_L[2], highest_list_L[3],highest_list_R[1], highest_list_R[2], highest_list_R[3]



        
    def fit(self, ds_X, ds_y):
        
        threshhold_L, index_L, ds_Y_L,threshhold_R, index_R, ds_Y_R  = self.split(ds_X, ds_y)

        self.model.append(Node(threshhold = threshhold_L, feature = index_L, ds_Y = ds_Y_L, index = index_L))
        self.model.append(Node(threshhold = threshhold_R, feature = index_R, ds_Y = ds_Y_R, index = index_R))
                               

        while self.max_depth > 0:
            threshhold_L, index_L, ds_Y_L,threshhold_R, index_R, ds_Y_R  = self.split(ds_X, ds_Y_L)
            threshhold_L, index_L, ds_Y_L,threshhold_R, index_R, ds_Y_R  = self.split(ds_X, ds_Y_R)
            
            #if self.stopping_condition() == True:
            #    self.model.append(Node(Leaf=True))
            #    self.max_depth = 0
            index = len(self.model)
            if self.max_depth == 1:
                self.model.append(Node(Leaf = True, ds_Y=ds_y))
                self.max_depth -= 1
            else:
                self.max_depth -= 1
                self.model.append(Node(threshhold = threshhold_L, feature = index_L, ds_Y = ds_Y_L, index = index+1))
                self.model.append(Node(threshhold = threshhold_R, feature = index_R, ds_Y = ds_Y_R, index = index+2))

            


    def predict(self, df_X):
        index: int = 0
        predictions: list = []
        for value in df_X:
            leaf = False
            index = 0
            while leaf == False:
                index_feature = self.model[index].feature
                


                next = self.model[index].evaluate(value[index_feature])

                if next == 'left':
                    index = self.model[index].next_index_L

                elif next == 'right':
                    index = self.model[index].next_index_R

                elif next == 'leaf':
                    predictions.append(self.model[index].class_dict)
                    leaf = True




        return predictions

        
        self.model(0).threshhold







        

            

            

                
                



In [ ]:

df_iris = pd.read_csv("iris.csv")

df_iris.drop('Id',axis=1,inplace=True)

df_iris_X = df_iris[['SepalLengthCm','SepalWidthCm','PetalLengthCm','PetalWidthCm']]

df_iris_y = df_iris.Species



flower_model = DecisionTree()

df_iris_X = df_iris_X.values.tolist()

flower_model.fit(df_iris_X, df_iris_y)



predict = (flower_model.predict(df_iris_X[:5]))
print(predict)
print(flower_model.model)
